# Overview of anomalies

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from anomaly.utils import AnomalyOverlapAnalyzer
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Constants

In [3]:
se_cols = [
    'mse',
    'mse_filter_250',
    # 'mse_filter_300',
    'mse_97',
    # 'mse_95',
    'mse_filter_250_97',
    # 'mse_filter_250_95',
    # 'mse_filter_300_97', 'mse_filter_300_95'
]

se_rank_cols = [
    f"rank_{col}" for col in se_cols
]

# ----------------------------------------------
rse_cols = [
    f"{col}_rel" for col in se_cols
]

rse_rank_cols = [
    f"rank_{col}_rel" for col in se_cols
]
# ----------------------------------------------
se_family = [
    'mse',
    'mse_97',
    # 'mse_95',
    'mse_filter_250',
    # 'mse_filter_300',
    'mse_filter_250_97',
    # 'mse_filter_250_95',
    # 'mse_filter_300_97', 'mse_filter_300_95'
]

rse_family = [
    f'{col}_rel' for col in se_family
]

# Custom functions

## IDs top anomalies

In [4]:
def get_ids(score, df, quantile=99, n_top=None, use_ntop=False):

    if use_ntop is False:
        
        quantile *= 0.01
        thresh = df[score].quantile(quantile)
        ids = set(df[df[score] > thresh].index)
        
    else:

        ids = set(
            df[score].sort_values(
                ascending=False
            ).iloc[:n_top].index
        )

    return ids

In [5]:
def top_unique_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    for score in scores_list:

        ids_top_dict[score] = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

    n_top = len(ids_top_dict[score])

    unique_ids_dict = AnomalyOverlapAnalyzer.get_unique_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    for score in scores_list:
        n_unique = len(unique_ids_dict[score])

        unique_pct = n_unique/n_top*100
        
        print(f"Unique to {score}:\n{n_unique} --> {unique_pct:.4f}%")

    return unique_ids_dict, ids_top_dict


In [6]:
def top_common_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    for score in scores_list:

        ids_top_dict[score] = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

    n_top = len(ids_top_dict[score])

    common_ids_dict = AnomalyOverlapAnalyzer.get_core_common_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    n_common = len(common_ids_dict)
    common_pct = n_common/n_top*100 
    print(f"N common:\n{n_common} -- > {common_pct:4f}%")

    return common_ids_dict, ids_top_dict

## Figures

In [7]:
def anomaly_plot(wave, specs, objids, ranks, save_to):

    fig, ax = plt.subplots(
        figsize=(10, 5)
    )

    for spec, objid, rank in zip(specs, objids, ranks):

        print(f'Rank {rank:03d}', end='\r')

        ax.clear()

        ax.plot(wave, spec, color="black", label=f'Rank: {rank}')

        ax.minorticks_on()
        ax.set_xlabel(r"$\lambda$ [nm]")
        ax.set_title(f"Object ID: {objid}")

        ax.legend(
            loc='upper left',
            frameon=False,
        )

        fig.savefig(
            f"{save_to}/{rank:03d}_{objid}.jpeg",
            bbox_inches='tight'
        )

    plt.close(fig)

# Config

## Directories

In [9]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
bin_id = 'bin_02'
#
ch_4_dir = f"{thesis_dir}/chapters/04_figures"
os.makedirs(f"{ch_4_dir}/{bin_id}", exist_ok=True)

## Data

In [10]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

In [11]:
score_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    index_col='specobjid'
)

rank = np.arange(score_df.shape[0]) + 1
score_rank_df = score_df.copy()
# score_rank_df
for col in score_df.columns:

    index_sorted = score_df.sort_values(
        by=col, ascending=False
    ).index

    score_rank_df.loc[index_sorted, f'rank_{col}'] = rank
    score_rank_df[f'rank_{col}'].astype(int)

n_spec = score_df.shape[0]
n_top_1_pct = int(n_spec*0.01)
n_top_1_pct, n_spec

(1818, 181850)

In [12]:
score = 'mse_97'
score_rank_df[[score, f'rank_{score}']].sort_values(
    by=score, ascending=False
).head(10)

,mse_97,rank_mse_97
specobjid,,
1414177610681837568,9.461991,1.0
808477533984024576,9.285621,2.0
1959115916855764992,8.363419,3.0
1621402570222233600,8.360323,4.0
1506455496133994496,8.104469,5.0
3089488382624032768,7.921358,6.0
2399384299896858624,7.707820,7.0
2013174804214999040,7.604570,8.0
1001066349105014784,7.550252,9.0


# Figs top anomalies

In [13]:
# ```python
n_top = 1000
all_scores = se_cols + rse_cols
plt.ioff()

for score in all_scores:

    specids_top_1 = score_df[score].sort_values(
        ascending=False
    ).index.to_numpy()[:n_top]

    ranks = np.zeros(n_top).astype(int)

    specs_top_1 = np.empty((n_top, wave.size))

    for i, objid in enumerate(specids_top_1):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs_top_1[i, :] = spectra[spec_idx, :]

        ranks[i] = i

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs_top_1,
        objids=specids_top_1, ranks=ranks,
        save_to=save_to
    )
# ```

# No free lunch theorem

## IDs per score

In [14]:
ids_top_dict = {}

all_scores = se_cols + rse_cols

for score in all_scores:

    ids_top_dict[score] = get_ids(
        score=score,
        df=score_df.copy(),
        quantile=99,
        n_top=1000,
        use_ntop=False
    )

n_top_1 = len(ids_top_dict[score])
n_top_1

1819

# Distinct IDs

## SE family

In [15]:
se_unique_ids_dict, se_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=se_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

Unique to mse:
499 --> 27.4327%
Unique to mse_filter_250:
312 --> 17.1523%
Unique to mse_97:
100 --> 5.4975%
Unique to mse_filter_250_97:
208 --> 11.4349%


In [16]:
score = 'mse'
unique_score_ids = list(se_unique_ids_dict[score])
score_rank_df.loc[
    unique_score_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse,rank_mse
specobjid,,
2034670801378109440,18.662506,105.0
1462649032210933760,17.966851,111.0
2408458845228656640,17.529365,117.0
2222540217548564480,16.427243,143.0
2796771996883511296,15.501871,161.0


### Figs unique per SE

In [17]:
plt.ioff()

for score in se_cols:

    specids = list(se_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_se/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

## RSE family

In [18]:
rse_unique_ids_dict, chi_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=rse_cols,
    quantile=99,
    # n_top=1000, use_ntop=True
)

Unique to mse_rel:
304 --> 16.7125%
Unique to mse_filter_250_rel:
193 --> 10.6102%
Unique to mse_97_rel:
152 --> 8.3562%
Unique to mse_filter_250_97_rel:
221 --> 12.1495%


In [19]:
score = 'mse_rel'
unique_res_ids = list(rse_unique_ids_dict[score])
score_rank_df.loc[
    unique_res_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse_rel,rank_mse_rel
specobjid,,
348037519964661760,21.213755,11.0
2186667001173272576,19.457949,14.0
2034670801378109440,15.924738,21.0
1462649032210933760,14.649748,24.0
1917559612572198912,14.373508,27.0


### Figs unique per RES

In [20]:
plt.ioff()

for score in rse_cols:

    specids = list(rse_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_rse/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

## All scores

In [21]:
all_scores = se_cols + rse_cols

all_unique_ids_dict, all_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=all_scores,
    quantile=99,
    # n_top=1000, use_ntop=True
)

Unique to mse:
342 --> 18.8015%
Unique to mse_filter_250:
178 --> 9.7856%
Unique to mse_97:
63 --> 3.4634%
Unique to mse_filter_250_97:
115 --> 6.3222%
Unique to mse_rel:
38 --> 2.0891%
Unique to mse_filter_250_rel:
119 --> 6.5421%
Unique to mse_97_rel:
86 --> 4.7279%
Unique to mse_filter_250_97_rel:
184 --> 10.1154%


In [22]:
score = 'mse'
unique_all_ids = list(all_unique_ids_dict[score])
score_rank_df.loc[
    unique_all_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False).head()

,mse,rank_mse
specobjid,,
2074044576338831360,13.637367,218.0
944761728170747904,11.446462,302.0
967167300304136192,11.097522,328.0
1848768656104777728,11.011874,334.0
1209267676514379776,10.832443,345.0


### Figs unique among all

In [23]:
all_scores = se_cols + rse_cols
plt.ioff()

for score in all_scores:

    specids = list(all_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_all/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

# Common IDs

## SE family

In [24]:
se_common_ids, se_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=se_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
561 -- > 30.841121%


In [25]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]
common_se_ids = list(se_common_ids)
score_rank_df.loc[
    common_se_ids, scores + rank_scores
].sort_values(by='mse', ascending=False).head()

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
1645099794210777088,156.687047,75.260937,6.592117,6.013027,1.0,1.0,25.0,52.0
2203533234665449472,65.204486,51.959115,5.145530,4.680116,8.0,2.0,152.0,415.0
1783527756395472896,51.386534,12.990448,5.828900,5.531697,13.0,43.0,60.0,94.0
2255301266211104768,49.194024,10.397455,6.868713,6.028563,16.0,91.0,17.0,49.0
1058509234703984640,42.354183,14.028320,4.881226,4.545068,20.0,31.0,247.0,578.0


### Figs common among SE

In [26]:
specids = list(se_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, f'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_ses"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)

## RSE Family

In [27]:
rse_common_ids, res_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=rse_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
757 -- > 41.616273%


In [28]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]

common_rse_ids = list(rse_common_ids)

score_rank_df.loc[
    common_rse_ids, scores + rank_scores
].sort_values(by='mse_rel', ascending=False).head()

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
1645099794210777088,156.687047,75.260937,6.592117,6.013027,1.0,1.0,25.0,52.0
2203533234665449472,65.204486,51.959115,5.145530,4.680116,8.0,2.0,152.0,415.0
429069599716173824,35.530281,31.050185,4.801460,4.444581,26.0,4.0,276.0,782.0
1427728542258456576,33.363459,28.438970,5.563669,5.436867,31.0,5.0,87.0,105.0
565203710213384192,33.223578,27.018231,5.539652,5.023883,32.0,6.0,89.0,212.0


### Figs common among RSE

In [29]:
specids = list(rse_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, f'rank_mse_rel'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_rses/"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)

## All scores

In [30]:
all_scores = se_cols + rse_cols 
all_common_ids, all_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=all_scores,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
320 -- > 17.592084%


In [31]:
scores = ['mse', 'mse_rel', 'mse_97', 'mse_97_rel']
rank_scores = [f'rank_{s}' for s in scores]

common_all_ids = list(all_common_ids)
score_rank_df.loc[
    common_all_ids, scores + rank_scores
].sort_values(by='mse', ascending=False).head(10)

,mse,mse_rel,mse_97,mse_97_rel,rank_mse,rank_mse_rel,rank_mse_97,rank_mse_97_rel
specobjid,,,,,,,,
1645099794210777088,156.687047,75.260937,6.592117,6.013027,1.0,1.0,25.0,52.0
2203533234665449472,65.204486,51.959115,5.145530,4.680116,8.0,2.0,152.0,415.0
1783527756395472896,51.386534,12.990448,5.828900,5.531697,13.0,43.0,60.0,94.0
2255301266211104768,49.194024,10.397455,6.868713,6.028563,16.0,91.0,17.0,49.0
3143664348946262016,40.987814,24.552827,5.459500,4.895498,21.0,8.0,95.0,259.0
833239361431562240,37.978053,11.685529,5.039013,5.038641,25.0,64.0,184.0,201.0
429069599716173824,35.530281,31.050185,4.801460,4.444581,26.0,4.0,276.0,782.0
1094476455129147392,35.427508,16.394939,6.491469,5.693062,27.0,19.0,28.0,76.0
2269865128344184832,34.960201,7.637110,5.406390,4.739681,28.0,333.0,106.0,357.0


### Figures

In [32]:
all_scores = se_cols + rse_cols
plt.ioff()


specids = list(all_common_ids)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, 'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_all"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)